# 01 — DistilBERT training

GPU-server notebook. Calls `CVSSTrainer` / `CWETrainer` from `veps.distilbert` — does **not** redefine training logic.

## 1. Setup

In [ ]:
from __future__ import annotations

import json
import sys

import pandas as pd
import torch
import transformers

from veps import paths as veps_paths
from veps.config import DistilBertHyperparameters, fetch_config_from_yaml
from veps.distilbert.base import get_device, set_seed
from veps.distilbert.cvss import CVSSTrainer
from veps.distilbert.cwe import CWETrainer

try:
    DB_CFG: DistilBertHyperparameters = fetch_config_from_yaml().distilbert
    print('[config] loaded from config.yaml')
except Exception as e:
    DB_CFG = DistilBertHyperparameters()
    print(f'[config] could not load config.yaml ({e}); using defaults')

for k, v in DB_CFG.model_dump().items():
    print(f'  {k}: {v}')

DEVICE = get_device()
print()
print(f'python:       {sys.version.split()[0]}')
print(f'torch:        {torch.__version__}')
print(f'transformers: {transformers.__version__}')
print(f'cuda avail:   {torch.cuda.is_available()}')
print(f'device:       {DEVICE}')

if not torch.cuda.is_available():
    raise RuntimeError(
        'CUDA not available. This notebook is for the GPU server.'
    )

gpu_idx = torch.cuda.current_device()
gpu_name = torch.cuda.get_device_name(gpu_idx)
gpu_mem = torch.cuda.get_device_properties(gpu_idx).total_memory / (1024 ** 3)
cap = torch.cuda.get_device_capability(gpu_idx)
print(f'gpu[{gpu_idx}]:     {gpu_name}  capability={cap[0]}.{cap[1]}  '
      f'total_memory={gpu_mem:.1f} GiB')

## 2. setup

In [ ]:
CVSS_DIR = veps_paths.TRAINING_DIR / 'cvss'
CWE_DIR  = veps_paths.TRAINING_DIR / 'cwe'
CVSS_METADATA = CVSS_DIR / 'cvss_metadata.csv'
CWE_METADATA  = CWE_DIR / 'cwe_metadata.csv'
MODELS_DIR = veps_paths.MODELS_DIR / 'distilbert'

for p in (CVSS_METADATA, CWE_METADATA):
    if not p.is_file():
        raise FileNotFoundError(
            f'{p} missing — run `veps extract-bert-trainset` first.'
        )
    print(f'ok: {p}')

MODELS_DIR.mkdir(parents=True, exist_ok=True)
_probe = MODELS_DIR / '.write_probe'
_probe.write_text('ok', encoding='utf-8')
_probe.unlink()
print(f'writable: {MODELS_DIR}')

def _split_counts(df, test_months, val_months):
    parsed = pd.to_datetime(df['published_date'], errors='coerce', utc=True, format='ISO8601')
    parsed = parsed.dropna().dt.tz_convert(None)
    max_date = parsed.max()
    test_start = max_date - pd.DateOffset(months=test_months)
    val_start  = test_start - pd.DateOffset(months=val_months)
    return {
        'train':      int((parsed <  val_start).sum()),
        'val':        int(((parsed >= val_start) & (parsed < test_start)).sum()),
        'test':       int((parsed >= test_start).sum()),
        'val_start':  val_start.date(),
        'test_start': test_start.date(),
    }

for label, p in (('cvss', CVSS_METADATA), ('cwe', CWE_METADATA)):
    df = pd.read_csv(p, usecols=['published_date'])
    c = _split_counts(df, DB_CFG.test_months, DB_CFG.val_months)
    print(f'[{label}] train={c["train"]}  val={c["val"]}  test={c["test"]}  '
          f'(val_start={c["val_start"]}  test_start={c["test_start"]})')

## 3. Run controls

Not hyperparameters — those live in `Config`. These select which model(s) to train and which seeds.

In [ ]:
MODEL = 'both'   # 'cvss' | 'cwe' | 'both'
NUM_SEEDS = 1

if MODEL not in ('cvss', 'cwe', 'both'):
    raise ValueError(f'MODEL must be cvss|cwe|both, got {MODEL!r}')
if NUM_SEEDS < 1:
    raise ValueError(f'NUM_SEEDS must be >= 1, got {NUM_SEEDS}')

base = list(DB_CFG.seeds) if DB_CFG.seeds else [42]
if NUM_SEEDS <= len(base):
    SEEDS = base[:NUM_SEEDS]
else:
    SEEDS = base + [base[-1] + i + 1 for i in range(NUM_SEEDS - len(base))]

print(f'MODEL:     {MODEL}')
print(f'NUM_SEEDS: {NUM_SEEDS}')
print(f'SEEDS:     {SEEDS}')

## 4. CVSS training

In [ ]:
cvss_metrics_per_seed = []
if MODEL in ('cvss', 'both'):
    for seed in SEEDS:
        print(f'\n=== cvss seed={seed} ===')
        set_seed(seed)
        cfg_seed = DB_CFG.model_copy(update={'seeds': [seed]})
        trainer = CVSSTrainer(data_dir=CVSS_DIR, models_dir=MODELS_DIR, config=cfg_seed)
        trainer.train()

        with open(veps_paths.cvss_run_manifest_path(MODELS_DIR)) as f:
            manifest = json.load(f)
        seed_metrics = (manifest.get('test_metrics') or [{}])[-1]
        cvss_metrics_per_seed.append({'seed': seed, **seed_metrics})

    df = pd.DataFrame(cvss_metrics_per_seed)
    print()
    print(df.to_string(index=False))
else:
    print('cvss training skipped (MODEL == cwe)')

## 5. CWE training

In [ ]:
cwe_metrics_per_seed = []
if MODEL in ('cwe', 'both'):
    for seed in SEEDS:
        print(f'\n=== cwe seed={seed} ===')
        set_seed(seed)
        cfg_seed = DB_CFG.model_copy(update={'seeds': [seed]})
        trainer = CWETrainer(data_dir=CWE_DIR, models_dir=MODELS_DIR, config=cfg_seed)
        trainer.train()

        with open(veps_paths.cwe_run_manifest_path(MODELS_DIR)) as f:
            manifest = json.load(f)
        seed_metrics = (manifest.get('test_metrics') or [{}])[-1]
        flat = {k: v for k, v in seed_metrics.items() if not isinstance(v, dict)}
        cwe_metrics_per_seed.append({'seed': seed, **flat})

    df = pd.DataFrame(cwe_metrics_per_seed)
    print()
    print(df.to_string(index=False))
else:
    print('cwe training skipped (MODEL == cvss)')